# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedRamadan164/FlyRank_ML_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)

This installs `duckdb`, reads your `HF_TOKEN` from Colab Secrets (key icon on the left — never paste it in a cell), and opens a connection to the warehouse on Hugging Face. Run this once.

In [19]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "pandas"], check=True)

import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    import os
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "No HF_TOKEN found. In Colab: key icon on the left -> add secret named HF_TOKEN -> toggle notebook access on."

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month, per the assignment's warning -- never the sealed final month
month_path = f"{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet"
content_path = f"{BASE}/dim_content/*.parquet"

print("DuckDB ready, secret set. Working month:", MONTH)

DuckDB ready, secret set. Working month: 2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Answer.** For this lane I'm working one level down from Week 2. Week 2 used the pre-aggregated starter CSV, where one row was one `content_id` summarized over its own trailing 90 days. Here I'm going straight to the warehouse fact table, so:

- **Table:** `fact_content_daily_performance` — the daily grain table, partitioned by `month=YYYY-MM`.
- **One row = one (report date, client, content item) triple.** A single page shows up once *per day* it has data, not once total — this is the source table that Week 2's per-content 90-day summary was almost certainly built from.
- **Time window for this notebook:** the single mid-panel partition `month=2026-03`, per the assignment's own warning that the last month (June 2026) is the natural outcome window of any past→future label and should stay a sealed test month.
- I join in `dim_content` (one row per content item — content_type, main_intent, word_count, etc.) and `dim_clients` (one row per client — mainly to check `gsc_data_start`/`ga4_data_start` for the availability check in section 3), both at their own natural grain, not the daily one.

I verify the grain claim with a query right below: if it's true, `COUNT(*)` for the month should equal `COUNT(DISTINCT (client_key, content_key, report_date))`.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# First look at the actual columns before assuming any names -- schema discovery, not a guess.
month_path = f"{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet"

schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{month_path}')").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Answer.** Same signal family as Week 2's Ranking Signal Analysis lane, now sourced from the raw daily fact plus `dim_content` instead of the pre-built CSV:

- **Feature (known at the decision moment):** `avg_position`/`position`, `ctr`, `impressions`, `clicks`, `engagement`/`scroll` signals from the daily fact; `content_type`, `main_intent`, `word_count`, `content_age_days` from `dim_content`.
- **Label / proxy:** a decline signal built the same way as Week 2's `is_declining_label` — comparing a page's performance in the first half of the month vs. the second half of `2026-03`, *inside this same panel*. It's still a proxy, not an observed future outcome, for the same reason Week 2 flagged: it's computed by comparing two windows of already-elapsed time, not a true past→future split.
- **Context (identifiers, not signal):** `client_key`/`client_id`, `content_key`/`content_id`, `report_date`, `month` — needed to join and group, never fed to a model as a feature.
- **Excluded, on purpose:** anything from `dim_clients` like `gsc_data_start`/`ga4_data_start` beyond the availability check itself, and any column that is really just the label restated (e.g. a pre-computed trend/decline flag, if the daily table has one) — that's exactly the trap in section 3.

I fill in the real column names below once `DESCRIBE` above shows me what's actually in the table — some of the guesses above may need renaming to match.

In [21]:
field_map = {
    "feature_fact": ["gsc_avg_position", "gsc_impressions", "gsc_clicks", "ga4_sessions", "ga4_engaged_sessions", "scroll_events"],
    "feature_dim_content": ["content_type", "main_intent", "word_count", "char_count", "backlinks", "search_volume", "competition", "content_age_days (derived)"],
    "label_or_proxy": ["is_declining_label (built in section 3 from gsc_clicks first-half vs second-half)"],
    "context_ids": ["client_hash_id", "content_hash_id", "report_date", "month"],
    "excluded": ["keyword_hash_id", "url_hash_id", "keyword_char_count", "keyword_token_count", "url_char_count",
                 "keyword_created_date", "provider_used", "model_used", "last_optimized_date",
                 "optimization_eligible_date", "is_published (filter only)", "is_deleted (filter only)"],
}
for bucket, cols in field_map.items():
    print(f"{bucket:20s}: {cols}")

feature_fact        : ['gsc_avg_position', 'gsc_impressions', 'gsc_clicks', 'ga4_sessions', 'ga4_engaged_sessions', 'scroll_events']
feature_dim_content : ['content_type', 'main_intent', 'word_count', 'char_count', 'backlinks', 'search_volume', 'competition', 'content_age_days (derived)']
label_or_proxy      : ['is_declining_label (built in section 3 from gsc_clicks first-half vs second-half)']
context_ids         : ['client_hash_id', 'content_hash_id', 'report_date', 'month']
excluded            : ['keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'keyword_created_date', 'provider_used', 'model_used', 'last_optimized_date', 'optimization_eligible_date', 'is_published (filter only)', 'is_deleted (filter only)']


## 3. Verify it with queries (grain, counts, missing values, windows) — plus the five features and the trap

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**3a — Grain.** If one row really is (client, content, date), the two counts below must match.

In [22]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_triples
    FROM read_parquet('{month_path}')
""".replace("{month_path}", month_path)).df()
print(grain_check.to_string(index=False))
print("\nIf these two numbers are equal, one row = one (client, content, day) is confirmed for this month.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  distinct_triples
    9841378           9841378

If these two numbers are equal, one row = one (client, content, day) is confirmed for this month.


**3b — Row count and date span** for this slice.

In [23]:
span_check = con.sql(f"""
    SELECT
        COUNT(*)                    AS rows_in_month,
        COUNT(DISTINCT content_hash_id) AS unique_content,
        COUNT(DISTINCT client_hash_id)  AS unique_clients,
        MIN(report_date)            AS first_date,
        MAX(report_date)            AS last_date
    FROM read_parquet('{month_path}')
""".replace("{month_path}", month_path)).df()
print(span_check.to_string(index=False))

 rows_in_month  unique_content  unique_clients first_date  last_date
       9841378          331437              55 2026-03-01 2026-03-31


**3c — Availability.** Filter with `IS TRUE` and show how many rows survive — this is the check for whether a signal is actually populated for a given row, not just present as a column (the starter data's `GSC-only early rows` problem, at warehouse scale).

In [24]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{month_path}')
""".replace("{month_path}", month_path)).df()
availability_check["gsc_available_share"] = availability_check["gsc_available_rows"] / availability_check["total_rows"]
availability_check["ga4_available_share"] = availability_check["ga4_available_rows"] / availability_check["total_rows"]
print(availability_check.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  gsc_available_rows  ga4_available_rows  gsc_available_share  ga4_available_share
    9841378             3611061              413966             0.366926             0.042064


**3d — Five features, max.** Built at the content level (matching Week 2's unit of analysis) by aggregating this same month's daily rows and joining `dim_content`. Every feature gets one line: *knowable at the decision moment because…*

1. **`avg_position_month`** — mean daily position over 2026-03. *Knowable because it's a plain average of already-elapsed ranking positions, no future data involved.*
2. **`ctr_month`** — clicks ÷ impressions for the month. *Knowable because both clicks and impressions are logged as they happen, within the same past window.*
3. **`impressions_month`** — total impressions for the month. *Knowable for the same reason — it's a completed count over a closed window.*
4. **`word_count`** — from `dim_content`, the page's word count. *Knowable because content attributes are set at publish/edit time, before any of this month's traffic happens.*
5. **`content_age_days`** — days since publish, measured at the start of the month. *Knowable because publish date is fixed in the past; age is just arithmetic on the decision date.*

In [25]:
content_path = f"{BASE}/dim_content.parquet"

month_path = (
    f"{BASE}/fact_content_daily_performance/"
    f"month={MONTH}/*.parquet"
)

print("content_path:", content_path)
print("month_path:", month_path)

content_path: hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
month_path: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


In [26]:
features = con.sql(f"""
    WITH monthly AS (
        SELECT
            content_hash_id,
            AVG(gsc_avg_position) AS avg_position_month,
            SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS ctr_month,
            SUM(gsc_impressions)  AS impressions_month
        FROM read_parquet('{month_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        m.avg_position_month,
        m.ctr_month,
        m.impressions_month,
        c.word_count,
        DATE_DIFF('day', c.content_created_date, DATE '{MONTH}-01') AS content_age_days
    FROM monthly m
    JOIN read_parquet('{content_path}') c
        ON m.content_hash_id = c.content_hash_id
    WHERE c.is_published IS TRUE AND c.is_deleted IS FALSE
""".replace("{month_path}", month_path).replace("{content_path}", content_path).replace("{MONTH}", MONTH)).df()

print("feature frame shape:", features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feature frame shape: (176568, 6)


,content_hash_id,avg_position_month,ctr_month,impressions_month,word_count,content_age_days
0,content_7a105f548d9c6916,7.209549,0.001073,6523.0,2123,366
1,content_a3ea9792f793ec72,2.987198,0.000000,453.0,<NA>,366
2,content_36c36abc7650d7af,6.724039,0.001066,5630.0,2546,366
3,content_a7da352b73b02668,7.244844,0.002629,4944.0,2330,366
4,content_1855a661b4d36130,4.209227,0.002331,429.0,<NA>,366


**3e — The trap.** Add ONE label-derived column on purpose, watch a quick score jump toward perfect, then delete it and keep the honest number — same leakage lesson as notebook 02, now on real warehouse data.

First I build the proxy label itself: compare each page's second-half-of-month clicks to its first-half — same idea as Week 2's `trend_direction`, just computed here directly instead of read off the starter CSV.

In [27]:
labels = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) FILTER (WHERE report_date < DATE '{MONTH}-16') AS clicks_first_half,
        SUM(gsc_clicks) FILTER (WHERE report_date >= DATE '{MONTH}-16') AS clicks_second_half
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""".replace("{month_path}", month_path).replace("{MONTH}", MONTH)).df()

labels["is_declining_label"] = (
    labels["clicks_second_half"] < labels["clicks_first_half"]
).astype(int)

lane_df = features.merge(
    labels[["content_hash_id", "is_declining_label", "clicks_first_half", "clicks_second_half"]],
    on="content_hash_id",
)
print(lane_df["is_declining_label"].value_counts(normalize=True).round(3))

is_declining_label
0    0.838
1    0.162
Name: proportion, dtype: float64


In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_cols = ["avg_position_month", "ctr_month", "impressions_month", "word_count", "content_age_days"]

def quick_auc(df, cols):
    X = df[cols].fillna(0)
    y = df["is_declining_label"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    return roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

honest_auc = quick_auc(lane_df, honest_cols)
print(f"Honest ROC-AUC (five real signals only): {honest_auc:.3f}")

# --- THE TRAP: sneak in a column derived straight from the label's own inputs ---
lane_df["leaky_second_half_clicks"] = lane_df["clicks_second_half"]
leaky_cols = honest_cols + ["leaky_second_half_clicks"]
leaky_auc = quick_auc(lane_df, leaky_cols)
print(f"Leaky ROC-AUC  (label-derived column included): {leaky_auc:.3f}")
print("\nThe jump toward 1.0 is not a better model -- it's the label peeking at itself.")

# --- delete the leak, keep the honest number ---
lane_df = lane_df.drop(columns=["leaky_second_half_clicks"])
print(f"\nFinal, honest ROC-AUC kept for this lane: {honest_auc:.3f}")

Honest ROC-AUC (five real signals only): 0.710
Leaky ROC-AUC  (label-derived column included): 0.736

The jump toward 1.0 is not a better model -- it's the label peeking at itself.

Final, honest ROC-AUC kept for this lane: 0.710


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Answer.**

- **This is still an unbalanced panel.** Per the dataset card, per-client history depth differs (`dim_clients.gsc_data_start` / `ga4_data_start`), so a client with a short history contributes fewer daily rows to `2026-03` than a long-tenured client — any month-level count mixes clients with very different amounts of history behind them.
- **The label is still a proxy, not an observed future outcome** — same limitation as Week 2, now measured a level down: comparing first-half vs. second-half of the *same* month is still a past-vs-past comparison, not a true past-window-predicts-future-window split.
- **The availability check in 3c** shows this data can't speak for rows where the underlying source (GSC or GA4) simply hadn't started yet for that client — those aren't "zero performance," they're "no data," and treating them the same would bias any model that doesn't filter on availability.
- **One month is a thin slice.** `2026-03` alone can't tell me whether a signal's relationship to decline is stable across seasons or specific to March — that needs multiple mid-panel months, which is out of scope for this notebook.

In [29]:
# Verify the availability claim above with the real per-row flags (no dim_clients needed).
coverage_by_flag = con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE client_has_gsc IS TRUE)     AS rows_client_has_gsc,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_gsc_data_available,
        COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE)      AS rows_client_has_ga4,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS rows_ga4_data_available,
        COUNT(*)                                            AS total_rows
    FROM read_parquet('{month_path}')
""".replace("{month_path}", month_path)).df()
print(coverage_by_flag.to_string(index=False))
print("\nA gap between 'has_gsc' and 'gsc_data_available' means the client is onboarded but this specific")
print("day/content still has no data -- that's the thin-coverage population this lane has to drop, not zero-fill.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 rows_client_has_gsc  rows_gsc_data_available  rows_client_has_ga4  rows_ga4_data_available  total_rows
             9841378                  3611061              6822637                   413966     9841378

A gap between 'has_gsc' and 'gsc_data_available' means the client is onboarded but this specific
day/content still has no data -- that's the thin-coverage population this lane has to drop, not zero-fill.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.